## 🎯 Learning Objectives
* Understand the critical role of dataset preparation in LLM finetuning.
* Identify common data formats suitable for LLM finetuning, particularly JSONL.
* Recognize key aspects of data quality (accuracy, relevance, diversity, consistency, cleanliness) and their impact.
* Evaluate dataset size considerations, balancing computational cost with model performance.
* Implement basic data cleaning and formatting techniques for instruction-tuned LLMs.


## FT01-L04: Dataset Preparation: Formats, Quality, and Size Considerations

Welcome to a crucial lesson in finetuning LLMs! Just as a chef needs high-quality ingredients to create a gourmet meal, an LLM needs a meticulously prepared dataset to learn and perform a specialized task effectively. The adage "garbage in, garbage out" is particularly true here. This lesson will equip you with the knowledge to prepare datasets that empower your LLMs to excel.

### The Foundation: Why Data Preparation is Paramount

Imagine you're teaching a brilliant student (your LLM) to become an expert in a niche field, say, summarizing legal documents. If you give them a textbook full of typos, irrelevant chapters, and inconsistent terminology, they'll struggle to grasp the core concepts and perform their job well. Similarly, a poorly prepared dataset can lead to:

*   **Suboptimal Performance**: The model fails to learn the desired patterns or generates irrelevant/incorrect outputs.
*   **Bias Amplification**: Existing biases in the data are learned and amplified by the model.
*   **Increased Training Costs**: Wasting computational resources on noisy or redundant data.
*   **Poor Generalization**: The model performs well on training data but fails on unseen, real-world examples.

### Data Formats: Speaking the LLM's Language

LLMs are trained on vast amounts of text, but for finetuning, we need to present data in a structured, consistent format that clearly defines the input and desired output. While various formats exist (CSV, Parquet, custom text files), **JSONL (JSON Lines)** has become the de-facto standard for instruction-tuned and conversational LLMs due to its flexibility and readability. Each line in a JSONL file is a self-contained JSON object, making it easy to stream and process.

Common JSONL structures for finetuning include:

1.  **Instruction-Output Pairs (e.g., Alpaca/Stanford style):**
    ```json
    {"instruction": "Summarize the following text:", "input": "[text to summarize]", "output": "[summary]"}
    ```
2.  **Conversational Turns (e.g., ShareGPT/OpenAI style):**
    ```json
    {"messages": [{"role": "user", "content": "[user query]"}, {"role": "assistant", "content": "[model response]"}]}
    ```

### Data Quality: The Gold Standard

Quality is not just about correctness; it encompasses several dimensions:

*   **Accuracy**: Is the information factually correct? Are there typos or grammatical errors? Incorrect labels or responses will mislead the model.
*   **Relevance**: Does the data directly pertain to the finetuning task? Including irrelevant examples dilutes the learning signal.
*   **Diversity**: Does the dataset cover a wide range of scenarios, edge cases, and linguistic variations relevant to the task? A diverse dataset helps the model generalize better and reduces bias.
*   **Consistency**: Is the style, tone, and format of the responses uniform? Inconsistent data can confuse the model about the desired output style.
*   **Cleanliness**: Is the data free from noise, duplicates, personally identifiable information (PII), or harmful content? Cleaning involves removing boilerplate text, HTML tags, special characters, and ensuring proper encoding.

**Strategies for Quality Improvement (2026 Perspective):**
*   **Automated Data Curation**: Leveraging LLMs themselves to identify low-quality examples, suggest improvements, or even generate synthetic data for augmentation.
*   **Active Learning**: Iteratively selecting the most informative examples for human annotation to maximize data efficiency.
*   **Human-in-the-Loop**: Combining automated checks with expert human review for critical datasets.

### Dataset Size: Finding the Sweet Spot

How much data do you need? This is a common question with no single answer, but here are guiding principles:

*   **Too Small**: If your dataset is too small (e.g., tens of examples), the model will likely **underfit**. It won't learn enough patterns and will perform poorly on new data. It might also overfit to the few examples it has, failing to generalize.
*   **Too Large**: While more data is generally better, there are diminishing returns. Extremely large datasets (millions of examples for specific tasks) can lead to significantly higher computational costs without proportional gains in performance. It also increases the complexity of data cleaning and validation.
*   **The "Goldilocks" Zone**: For LoRA/QLoRA finetuning, you can often achieve impressive results with hundreds to tens of thousands of high-quality examples. The exact number depends on the complexity of the task, the base model's capabilities, and the desired performance.

**General Rule of Thumb:** Start with a smaller, high-quality dataset (e.g., 500-5000 examples). Iterate and expand if performance plateaus or if you identify specific areas where the model struggles due to lack of diverse examples.

### Step-by-Step Data Preparation Process

1.  **Define the Task & Target Output**: Clearly articulate what you want the LLM to do and what its output should look like.
2.  **Identify Data Sources**: Where can you find relevant raw data? (e.g., internal documents, public datasets, web scraping, synthetic generation).
3.  **Collect Raw Data**: Gather the data from identified sources.
4.  **Clean & Preprocess**: Remove noise, handle missing values, normalize text, remove PII.
5.  **Format for LLM**: Transform the cleaned data into the chosen finetuning format (e.g., JSONL with `messages` structure).
6.  **Split Data**: Divide into training, validation, and test sets. A common split is 80/10/10 or 90/5/5.

Let's dive into a practical example of preparing a dataset for an instruction-tuned LLM.


In [ ]:
import json
import random
import re
from datasets import Dataset # Hugging Face datasets library for easy handling

# --- 1. Define a synthetic dataset generation function ---
# This simulates collecting raw, slightly noisy data
def generate_synthetic_data(num_samples=100):
    data = []
    product_types = ["smartphone", "laptop", "smartwatch", "headphones", "tablet"]
    sentiments = ["positive", "negative", "neutral"]
    
    positive_feedback = [
        "Absolutely love this product! It's fast and reliable.",
        "Fantastic device, exceeded my expectations. Highly recommend.",
        "Great value for money, battery life is amazing.",
        "Sleek design and powerful performance. A joy to use.",
        "Best purchase this year. Flawless experience."
    ]
    
    negative_feedback = [
        "Very disappointed. The battery drains too quickly.",
        "Poor quality, broke after a month. Avoid at all costs.",
        "Laggy performance and constant crashes. Unusable.",
        "Overpriced for what it offers. Expected more.",
        "Customer service was unhelpful. Product itself is mediocre."
    ]
    
    neutral_feedback = [
        "It's okay, does the job. Nothing spectacular.",
        "Average performance, no major complaints or praises.",
        "Works as advertised, but could be better.",
        "Decent product for the price point.",
        "Just another gadget, nothing revolutionary."
    ]
    
    for i in range(num_samples):
        product = random.choice(product_types)
        sentiment = random.choice(sentiments)
        
        if sentiment == "positive":
            review = random.choice(positive_feedback)
        elif sentiment == "negative":
            review = random.choice(negative_feedback)
        else:
            review = random.choice(neutral_feedback)
            
        # Introduce some noise for cleaning demonstration
        if random.random() < 0.1: # 10% chance of a very short/irrelevant review
            review = random.choice(["bad", "good", "meh", "This is spam.", """<p>HTML content</p>"""])
        if random.random() < 0.15: # 15% chance of extra whitespace/special chars
            review = f"  {review} !!! "
        if random.random() < 0.05: # 5% chance of PII-like data
            review = f"{review} My email is user@example.com and phone is 123-456-7890."
            
        data.append({
            "id": i,
            "product_type": product,
            "review_text": review,
            "sentiment_label": sentiment
        })
    return data

# --- 2. Data Cleaning and Preprocessing Function ---
def clean_and_format_for_finetuning(raw_data):
    processed_data = []
    
    for item in raw_data:
        review = item["review_text"]
        
        # Step 1: Remove HTML tags (basic example)
        review = re.sub(r'<.*?>', '', review)
        
        # Step 2: Remove PII (email, phone numbers - basic regex)
        review = re.sub(r'\S*@\S*|\d{3}[-\s]?\d{3}[-\s]?\d{4}', '[REDACTED]', review)
        
        # Step 3: Remove extra whitespace and special characters (keep punctuation)
        review = re.sub(r'[^a-zA-Z0-9.,?!\s]', '', review) # Keep alphanumeric, common punctuation
        review = re.sub(r'\s+', ' ', review).strip() # Replace multiple spaces with single, strip leading/trailing
        
        # Step 4: Filter out very short or irrelevant reviews after cleaning
        if len(review) < 10 or review.lower() in ["bad", "good", "meh", "this is spam", "[redacted]"]:
            continue # Skip this item if it's too short or generic after cleaning
            
        # Step 5: Format into conversational JSONL structure for instruction tuning
        # Task: Classify the sentiment of a product review.
        instruction = f"Classify the sentiment of the following {item['product_type']} review as positive, negative, or neutral."
        
        # The desired output from the LLM
        target_output = item["sentiment_label"].capitalize()
        
        # Create the message list for the LLM
        messages = [
            {"role": "user", "content": f"{instruction}\nReview: {review}"},
            {"role": "assistant", "content": target_output}
        ]
        
        processed_data.append({"messages": messages})
        
    return processed_data

# --- Main Execution --- 
if __name__ == "__main__":
    print("Generating synthetic raw data...")
    raw_dataset = generate_synthetic_data(num_samples=200) # Generate 200 samples
    print(f"Generated {len(raw_dataset)} raw samples.")
    
    print("\nExample of raw data (first 2 samples):")
    for i in range(min(2, len(raw_dataset))):
        print(json.dumps(raw_dataset[i], indent=2))
        
    print("\nCleaning and formatting data for finetuning...")
    finetuning_dataset = clean_and_format_for_finetuning(raw_dataset)
    print(f"Processed {len(finetuning_dataset)} samples after cleaning and filtering.")
    
    print("\nExample of formatted finetuning data (first 2 samples):")
    for i in range(min(2, len(finetuning_dataset))):
        print(json.dumps(finetuning_dataset[i], indent=2))
        
    # --- 3. Save the formatted data to a JSONL file ---
    output_filename = "finetuning_data.jsonl"
    
    # Using Hugging Face's `datasets` library for robust saving
    # First, convert list of dicts to a Hugging Face Dataset object
    hf_dataset = Dataset.from_list(finetuning_dataset)
    hf_dataset.to_json(output_filename, orient="records", lines=True, force_ascii=False)
    
    print(f"\nFormatted dataset saved to '{output_filename}'.")
    
    # Verify by loading a few lines
    print(f"\nVerifying content of '{output_filename}' (first 3 lines):")
    with open(output_filename, 'r', encoding='utf-8') as f:
        for _ in range(min(3, len(finetuning_dataset))):
            print(f.readline().strip())


### Interpreting the Code Output and Practical Considerations

The code above demonstrates a simplified yet illustrative workflow for preparing a dataset for LLM finetuning. Let's break down what you've seen and its implications:

1.  **Synthetic Data Generation**: We started by creating `generate_synthetic_data`. In a real-world scenario, this step would involve collecting data from various sources like customer support logs, public forums, internal documents, or even using advanced techniques like **synthetic data generation via another LLM** (a growing trend in 2026 to augment scarce human-labeled data). The synthetic data intentionally included noise (short reviews, HTML, PII, extra spaces) to mimic real-world imperfections.

2.  **Cleaning and Preprocessing (`clean_and_format_for_finetuning`)**:
    *   **HTML Removal**: `re.sub(r'<.*?>', '', review)` is a basic regex to strip HTML tags. For complex HTML, libraries like `BeautifulSoup` would be more robust.
    *   **PII Redaction**: `re.sub(r'\S*@\S*|\d{3}[-\s]?\d{3}[-\s]?\d{4}', '[REDACTED]', review)` demonstrates basic PII removal for emails and phone numbers. In production, this is a critical and much more complex task, often involving specialized NLP models or services to ensure compliance (e.g., GDPR, HIPAA).
    *   **Text Normalization**: Removing extra whitespace and non-standard characters (`re.sub(r'\s+', ' ', review).strip()`) ensures consistency and reduces noise. This step is crucial for the model to focus on meaningful tokens.
    *   **Filtering**: `if len(review) < 10 or review.lower() in [...]` shows how to remove low-quality or irrelevant examples. This is a vital step to prevent the model from learning from 
